# Day 01 · Python, the AI dialect

**Goal:** by the end of this notebook you can load a CSV, clean it, aggregate it and report on it using *pure Python* — and you can read the five idioms (f-strings, comprehensions, lambdas, `try/except`, `with open`) that every AI training script is written in.

**Run in Google Colab:** File → Open notebook → Upload → pick this `.ipynb`. Then *Runtime → Run all* (or Shift+Enter cell by cell).

**Runtime:** CPU is fine. No GPU needed today.

| Section | What you'll do |
|---|---|
| 1 | Names are stickers: mutability, aliasing, copies |
| 2 | The four containers (list / tuple / dict / set) |
| 3 | Control flow + functions with keyword defaults |
| 4 | The five idioms: f-strings, comprehensions, lambda, errors, files |
| 5 | Build: **salary-vs-skills analyzer** from a CSV (pure Python) |
| 6 | Modules, `__main__`, `pip` |
| 7 | 🎯 Try it yourself + Homework starter |

## 0. Setup

Everything today is in the Python standard library, so there is nothing to `pip install`. We just print the Python version so you know what you're running (Colab ships Python 3.11+).

In [23]:
import sys, platform
print("Python", sys.version.split()[0], "on", platform.system())
print("Nothing to install today — the standard library is enough.")

Python 3.13.5 on Windows
Nothing to install today — the standard library is enough.


## 1. Names are stickers, not boxes 🏷️

In C, a variable is a **box** that holds a value. In Python a variable is a **sticker** you put on an object. `b = a` does not copy anything — it puts a second sticker on the same object. This is the number-one source of "why did my data change?!" bugs in ML preprocessing code.

Watch it happen. The built-in `id()` returns the memory address of an object; two names with the same `id` point at the same thing.

In [24]:
a = [1, 2, 3]
b = a            # second sticker on the SAME list
b.append(4)

print("a =", a)          # a changed too!
print("b =", b)
print("same object?", a is b, "| id(a) == id(b):", id(a) == id(b))
print('is a == b ',a == b)

a = [1, 2, 3, 4]
b = [1, 2, 3, 4]
same object? True | id(a) == id(b): True
is a == b  True


In [25]:
# To get an independent copy, ask for one explicitly
c = a.copy()     # or a[:]  or list(a)
print("same object?", a is c)
c.append(99)
print("a =", a)
print("c =", c)
print("same object?", a is c)

same object? False
a = [1, 2, 3, 4]
c = [1, 2, 3, 4, 99]
same object? False


**Immutable objects behave differently.** Ints, floats, strings and tuples can never change in place. `y = y + 1` creates a *new* int and moves the `y` sticker to it; `x` is untouched.

In [26]:
x = 5
y = x
y = y + 1                    # new object; y re-bound
print("x =", x, "| y =", y, "| same object?", x is y)

s = "chai"
t = s.upper()                # strings are immutable: .upper() returns a NEW string
print(s, "->", t)

shape = (28, 28, 3)          # tuples are immutable — perfect for things that must not change
try:
    shape[0] = 32
except TypeError as e:
    print("TypeError:", e)

x = 5 | y = 6 | same object? False
chai -> CHAI
TypeError: 'tuple' object does not support item assignment


> 🧠 **Interview Q:** *"Is Python pass-by-value or pass-by-reference?"*
> Neither — it is **pass-by-object** (pass-by-assignment). A function receives another sticker on the same object. If the object is mutable (list, dict, NumPy array, PyTorch tensor) the function can change it in place; if immutable (int, str, tuple) it cannot.

Let's prove it with the classic *mutable default argument* trap — a real bug that shows up in interviews.

In [27]:
def add_loss_bad(value, history=[]):      # ⚠️ the default list is created ONCE, at def time
    history.append(value)
    return history

print(add_loss_bad(0.9))
print(add_loss_bad(0.7))     # surprise: the previous 0.9 is still there!

def add_loss_good(value, history=None):   # ✅ the idiom: default None, create inside
    if history is None:
        history = []
    history.append(value)
    return history

print(add_loss_good(0.9))
print(add_loss_good(0.7))

[0.9]
[0.9, 0.7]
[0.9]
[0.7]


## 2. The four containers you'll meet in every AI script 📦

| Container | Ordered? | Mutable? | Use it for |
|---|---|---|---|
| `list [ ]` | ✅ | ✅ | things that grow: losses per epoch, a batch of samples |
| `tuple ( )` | ✅ | ❌ | fixed records: image `shape`, `(x, y)` pairs, multiple return values |
| `dict { k: v }` | insertion order | ✅ | lookups by name: configs, JSON, `word → index` vocab |
| `set { }` | ❌ | ✅ | unique items & membership: "which skills appear anywhere?" |

In [28]:
losses = [0.92, 0.71, 0.55, 0.43]           # list: grows every epoch
losses.append(0.38)
print("last two losses:", losses[-2:], "| best:", min(losses))

image_shape = (224, 224, 3)                   # tuple: height, width, channels — never changes
h, w, ch = image_shape                        # tuple unpacking
print(f"{h}x{w} image with {ch} channels")

config = {"lr": 0.01, "epochs": 5, "model": "resnet18"}   # dict: hyperparameters by name
config["batch_size"] = 32                     # add a key
print("lr =", config["lr"], "| keys:", list(config))
print("optimizer:", config.get("optimizer", "adam (default)"))   # .get with a default = no KeyError

jd1 = {"python", "sql", "pandas"}             # set: unique, unordered
jd2 = {"python", "pytorch", "docker"}
print("in both JDs:", jd1 & jd2)
print("in any JD:  ", jd1 | jd2)
print("only in jd2:", jd2 - jd1)

last two losses: [0.43, 0.38] | best: 0.38
224x224 image with 3 channels
lr = 0.01 | keys: ['lr', 'epochs', 'model', 'batch_size']
optimizer: adam (default)
in both JDs: {'python'}
in any JD:   {'sql', 'pytorch', 'python', 'docker', 'pandas'}
only in jd2: {'docker', 'pytorch'}


## 3. Control flow and functions 🚦

Indentation **is** the syntax. A PyTorch training loop is literally `for epoch in range(N):` containing `for batch in loader:`. Learn to read nested loops as a story: *"for every epoch, for every batch, do a step."*

Functions have **keyword arguments with defaults** — this is how every library API is designed (`LinearRegression(fit_intercept=True)`, `Adam(params, lr=1e-3)`).

In [29]:
def train(model_name, data, epochs=3, lr=0.01, verbose=True):
    """Fake training loop: returns a list of losses (one per epoch)."""
    losses = []
    loss = 1.0
    for epoch in range(epochs):              # outer loop: epochs
        for batch in data:                   # inner loop: batches
            loss = loss * (1 - lr * len(batch) / 10)   # pretend the loss shrinks
        losses.append(round(loss, 4))
        if verbose:
            print(f"[{model_name}] epoch {epoch + 1}/{epochs}  loss={loss:.4f}")
    return losses

fake_data = [[1, 2, 3], [4, 5], [6, 7, 8, 9]]
hist = train("tiny-net", fake_data)                    # defaults used
hist2 = train("tiny-net", fake_data, epochs=5, lr=0.05, verbose=False)   # keyword args — readable!
print("5-epoch run:", hist2)

[tiny-net] epoch 1/3  loss=0.9910
[tiny-net] epoch 2/3  loss=0.9821
[tiny-net] epoch 3/3  loss=0.9733
5-epoch run: [0.9556, 0.9133, 0.8728, 0.834, 0.7971]


In [30]:
# Functions are objects: you can put them in a dict and pick one by name.
# This exact pattern appears in every ML config system ("activation": "relu").
import math

activations = {
    "relu":    lambda z: max(0.0, z),
    "sigmoid": lambda z: 1 / (1 + math.exp(-z)),
    "tanh":    math.tanh,                   # a built-in function, passed as a value
}
for name, fn in activations.items():
    print(f"{name:8} f(-1.0) = {fn(-1.0):7.4f}   f(2.0) = {fn(2.0):7.4f}")

relu     f(-1.0) =  0.0000   f(2.0) =  2.0000
sigmoid  f(-1.0) =  0.2689   f(2.0) =  0.8808
tanh     f(-1.0) = -0.7616   f(2.0) =  0.9640


## 4. The five idioms that make AI code look like AI code ✨

### 4a. f-strings and format specs
`f"{loss:8.4f}"` → fixed-point float, 4 decimals, padded to width 8. `:.1%` → percent. `:,` → thousands separators. `!r` → repr with quotes (debugging).

In [31]:
loss, acc, n, name = 0.0312345, 0.9376, 1250000, "roll no. 42"
print(f"loss={loss:.4f}  acc={acc:.1%}  samples={n:,}  who={name!r}")
print(f"|{'skill':<12}|{'score':>8}|")        # < left-align, > right-align, width
print(f"|{'python':<12}|{0.87:>8.2f}|")
print(f"{'-'*23}")                            # string repetition
print(f"2 + 2 = {2 + 2}, {'yes' if acc > 0.9 else 'no'}")   # any expression works inside {}

loss=0.0312  acc=93.8%  samples=1,250,000  who='roll no. 42'
|skill       |   score|
|python      |    0.87|
-----------------------
2 + 2 = 4, yes


### 4b. Comprehensions
Read `[expression for item in source if condition]` as a pipeline: **source → filter → transform → collect**. It is the for-loop you already know, folded into one line.

In [32]:
nums = list(range(1, 11))

# for-loop version
squares_of_evens = []
for n in nums:
    if n % 2 == 0:
        squares_of_evens.append(n ** 2)

# comprehension version — same result, one line, slightly faster
squares_of_evens_2 = [n ** 2 for n in nums if n % 2 == 0]
print(squares_of_evens, squares_of_evens_2, squares_of_evens == squares_of_evens_2)

# dict comprehension: the word->index vocabulary every NLP model starts with
vocab = ["chai", "samosa", "biryani", "dosa"]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}     # invert it
print(word2idx, idx2word)

# set comprehension + nested comprehension (flatten a list of lists)
batches = [[1, 2], [3, 4, 5], [6]]
flat = [x for batch in batches for x in batch]       # read left to right: for batch..., for x...
print(flat, {x % 3 for x in flat})

[4, 16, 36, 64, 100] [4, 16, 36, 64, 100] True
{'chai': 0, 'samosa': 1, 'biryani': 2, 'dosa': 3} {0: 'chai', 1: 'samosa', 2: 'biryani', 3: 'dosa'}
[1, 2, 3, 4, 5, 6] {0, 1, 2}


### 4c. `lambda` — a tiny nameless function, used once
Most common use: as the `key=` of `sorted`/`min`/`max`, and later as `df.apply(lambda row: ...)`.

In [33]:
candidates = [("Priya", 7.8, 2), ("Arjun", 8.4, 0), ("Meera", 9.1, 1)]   # (name, CGPA, internships)

print("by CGPA desc:      ", sorted(candidates, key=lambda c: c[1], reverse=True))
print("by internships:    ", sorted(candidates, key=lambda c: c[2]))
print("best CGPA:         ", max(candidates, key=lambda c: c[1])[0])

# lambda vs def — identical behaviour; def is better once it needs a name or >1 line
square = lambda z: z * z
def square_def(z):
    return z * z
print(square(7), square_def(7))

by CGPA desc:       [('Meera', 9.1, 1), ('Arjun', 8.4, 0), ('Priya', 7.8, 2)]
by internships:     [('Arjun', 8.4, 0), ('Meera', 9.1, 1), ('Priya', 7.8, 2)]
best CGPA:          Meera
49 49


### 4d. Errors: the difference between a script and a tool
A script crashes at row 3,417 because the salary column says `"Not disclosed"`. A tool catches that, counts it and keeps going. **Catch the specific exception you expect.** Never write a bare `except:`.

In [34]:
raw = ["12.5", "8", "Not disclosed", "15.0", "", "22"]

parsed, skipped = [], 0
for value in raw:
    try:
        parsed.append(float(value))
    except ValueError:                      # only the error we expect
        skipped += 1
    else:                                   # runs only if no exception
        pass
    finally:                                # runs always (cleanup goes here)
        pass
print(f"parsed={parsed}  skipped={skipped}")

# Raising your own errors is how libraries tell you that you mis-used them
def safe_divide(a, b):
    if b == 0:
        raise ValueError("b must be non-zero")
    return a / b

for b in (4, 0):
    try:
        print(safe_divide(10, b))
    except ValueError as e:
        print("caught:", e)

parsed=[12.5, 8.0, 15.0, 22.0]  skipped=2
2.5
caught: b must be non-zero


### 4e. Files: `with open(...)` guarantees the file closes, even if an error happens
Reading datasets, writing logs, saving checkpoints — all use this shape. We'll write a small file, read it back, and also use `json` (the format every API and config file speaks).

In [35]:
import json, os

# write
with open("notes.txt", "w", encoding="utf-8") as f:
    f.write("day 01: python basics\n")
    f.write("day 02: oop, generators, decorators\n")

# read — three ways
with open("notes.txt", encoding="utf-8") as f:
    whole = f.read()
with open("notes.txt", encoding="utf-8") as f:
    lines = [line.strip() for line in f]          # a file object is iterable, line by line
print(whole)
print(lines)

# JSON: dict <-> text
cfg = {"lr": 0.01, "epochs": 5, "tags": ["baseline", "day01"]}
with open("config.json", "w") as f:
    json.dump(cfg, f, indent=2)
with open("config.json") as f:
    cfg_back = json.load(f)
print(cfg_back == cfg, "| file size:", os.path.getsize("config.json"), "bytes")

day 01: python basics
day 02: oop, generators, decorators

['day 01: python basics', 'day 02: oop, generators, decorators']
True | file size: 83 bytes


## 5. Build: the salary-vs-skills analyzer (pure Python) 💼

A CSV of job postings with `title, company, city, salary_lpa, skills`. Questions a recruiter's data team would actually ask:
1. Which skills are asked for most often?
2. Which skill is associated with the **highest average salary**?
3. What does the market look like per city?

No pandas yet — we build it with today's tools so that on Day 03 you can *see* what pandas is replacing.

First we generate a realistic synthetic dataset (with a few deliberately messy rows) and save it as `jobs.csv`.

In [36]:
import csv, random
random.seed(42)                                     # reproducible

titles = ["Data Analyst", "ML Engineer", "AI Engineer", "Data Scientist", "Python Developer", "NLP Engineer"]
cities = ["Bengaluru", "Hyderabad", "Pune", "Gurugram", "Chennai", "Noida", "Remote"]
companies = ["Zepto", "Flipkart", "Razorpay", "Infosys", "TCS", "Swiggy", "PhonePe", "Freshworks", "Ola", "Zomato"]
skill_pool = {
    "python": 4.0, "sql": 1.0, "pandas": 1.5, "numpy": 1.2, "scikit-learn": 2.0, "pytorch": 4.5,
    "tensorflow": 3.5, "docker": 2.5, "aws": 3.0, "excel": -1.0, "tableau": 0.5, "llms": 5.0,
    "langchain": 4.0, "fastapi": 2.5, "git": 0.8, "statistics": 1.5, "spark": 3.0, "nlp": 3.0,
}
base = {"Data Analyst": 5, "ML Engineer": 11, "AI Engineer": 12, "Data Scientist": 10, "Python Developer": 7, "NLP Engineer": 11}

rows = []
for i in range(600):
    t = random.choice(titles)
    skills = random.sample(list(skill_pool), k=random.randint(3, 7))
    if random.random() < 0.85: skills = list({"python", *skills})     # python in most postings
    salary = base[t] + sum(skill_pool[s] for s in skills) * 0.7 + random.gauss(0, 1.5)
    salary_txt = f"{max(salary, 3):.1f}"
    if random.random() < 0.04: salary_txt = "Not disclosed"           # messy rows on purpose
    rows.append({"title": t, "company": random.choice(companies), "city": random.choice(cities),
                 "salary_lpa": salary_txt, "skills": "; ".join(skills)})

with open("jobs.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader(); w.writerows(rows)

print("wrote jobs.csv with", len(rows), "rows. First 3:")
for r in rows[:3]: print(r)

wrote jobs.csv with 600 rows. First 3:
{'title': 'NLP Engineer', 'company': 'Flipkart', 'city': 'Chennai', 'salary_lpa': '17.5', 'skills': 'docker; aws; python'}
{'title': 'Data Scientist', 'company': 'Ola', 'city': 'Hyderabad', 'salary_lpa': '14.1', 'skills': 'pandas; tensorflow; python'}
{'title': 'NLP Engineer', 'company': 'Infosys', 'city': 'Remote', 'salary_lpa': '22.2', 'skills': 'git; fastapi; excel; scikit-learn; python; langchain; docker'}


### 5a. Load + clean (with a guard for bad rows)
`csv.DictReader` gives one dict per row. We convert salary to `float` (skipping "Not disclosed") and turn the `skills` string into a **set** of lowercase skills.

In [37]:
def load_jobs(path):
    """Return (clean_rows, n_skipped). Each row has float salary and a set of skills."""
    clean, skipped = [], 0
    with open(path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            try:
                r["salary_lpa"] = float(r["salary_lpa"])
            except ValueError:
                skipped += 1
                continue
            r["skills"] = {s.strip().lower() for s in r["skills"].split(";") if s.strip()}
            clean.append(r)
    return clean, skipped

jobs, n_skipped = load_jobs("jobs.csv")
print(f"loaded {len(jobs):,} rows, skipped {n_skipped} with bad salary")
print(jobs[0])

loaded 572 rows, skipped 28 with bad salary
{'title': 'NLP Engineer', 'company': 'Flipkart', 'city': 'Chennai', 'salary_lpa': 17.5, 'skills': {'docker', 'aws', 'python'}}


### 5b. Aggregate: most-demanded skills and highest-paying skills
`collections.Counter` counts hashable things; `most_common()` sorts for you. Average-salary-per-skill is a dict comprehension over the set of all skills.

In [38]:
from collections import Counter

# Q1: most-demanded skills
demand = Counter(s for r in jobs for s in r["skills"])
print("Top 8 demanded skills:")
for skill, n in demand.most_common(8):
    print(f"  {skill:<14}{n:>5}  {'█' * (n // 20)}")

# Q2: average salary per skill (only skills that appear in >= 20 postings)
all_skills = set().union(*[r["skills"] for r in jobs])
avg_by_skill = {
    s: sum(r["salary_lpa"] for r in jobs if s in r["skills"]) / demand[s]
    for s in all_skills if demand[s] >= 20
}
print("\nHighest-paying skills (avg LPA):")
for skill, avg in sorted(avg_by_skill.items(), key=lambda kv: kv[1], reverse=True)[:8]:
    print(f"  {skill:<14}{avg:6.1f} LPA")

Top 8 demanded skills:
  python          505  █████████████████████████
  spark           188  █████████
  docker          178  ████████
  excel           175  ████████
  scikit-learn    173  ████████
  langchain       170  ████████
  tensorflow      164  ████████
  pytorch         163  ████████

Highest-paying skills (avg LPA):
  llms            21.3 LPA
  pytorch         21.2 LPA
  nlp             20.7 LPA
  langchain       20.7 LPA
  spark           20.5 LPA
  tensorflow      20.3 LPA
  aws             20.2 LPA
  fastapi         20.2 LPA


In [39]:
# Q3: per-city picture (dict of city -> list of salaries, then summarise)
by_city = {}
for r in jobs:
    by_city.setdefault(r["city"], []).append(r["salary_lpa"])     # setdefault: create list on first sight

print(f"{'city':<12}{'jobs':>6}{'avg LPA':>10}{'max LPA':>10}")
for city, sal in sorted(by_city.items(), key=lambda kv: -sum(kv[1]) / len(kv[1])):
    print(f"{city:<12}{len(sal):>6}{sum(sal)/len(sal):>10.1f}{max(sal):>10.1f}")

city          jobs   avg LPA   max LPA
Gurugram        79      20.0      29.5
Bengaluru      101      19.6      27.4
Chennai         84      19.5      27.3
Pune            85      19.2      30.2
Noida           69      19.2      27.6
Remote          82      19.2      28.5
Hyderabad       72      18.7      28.4


> 💼 **Career note:** this is exactly the shape of a 30-minute screening task for *Data Analyst / Junior ML Engineer* roles: load → clean → aggregate → sort → report. On Day 03 the same three questions become three `groupby` lines in pandas — but you'll know what those lines are doing.

## 6. Modules, `__main__` and `pip` 📦

A **module** is any `.py` file. A **package** is a folder of modules. `pip install` downloads a package once; `import` loads it into the current program.

Colab lets us write a file with the `%%writefile` magic, then import it like a library *and* run it like a script.

In [40]:
%%writefile jobs_tool.py
"""Tiny job-market analyzer.  Usage:  python jobs_tool.py jobs.csv [--city Pune]"""
import sys, csv, json
from collections import Counter

def load(path):
    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            try:
                r["salary_lpa"] = float(r["salary_lpa"])
            except ValueError:
                continue
            r["skills"] = {s.strip().lower() for s in r["skills"].split(";") if s.strip()}
            rows.append(r)
    return rows

def report(rows, top_n=5):
    counts = Counter(s for r in rows for s in r["skills"])
    avg = sum(r["salary_lpa"] for r in rows) / len(rows) if rows else 0.0
    return {"n_jobs": len(rows), "avg_salary_lpa": round(avg, 2), "top_skills": counts.most_common(top_n)}

if __name__ == "__main__":                       # only runs when executed directly, not on import
    rows = load(sys.argv[1])
    if "--city" in sys.argv:
        city = sys.argv[sys.argv.index("--city") + 1]
        rows = [r for r in rows if r["city"].lower() == city.lower()]
    print(json.dumps(report(rows), indent=2))

Overwriting jobs_tool.py


In [41]:
# 1) use it as a library
import importlib, jobs_tool
importlib.reload(jobs_tool)                       # in case you edit and re-run the cell above
print(jobs_tool.report(jobs_tool.load("jobs.csv"), top_n=3))

# 2) use it as a command-line script (the ! runs a shell command in Colab)
!python jobs_tool.py jobs.csv --city Pune

{'n_jobs': 572, 'avg_salary_lpa': 19.35, 'top_skills': [('python', 505), ('spark', 188), ('docker', 178)]}
{
  "n_jobs": 85,
  "avg_salary_lpa": 19.23,
  "top_skills": [
    [
      "python",
      77
    ],
    [
      "tableau",
      31
    ],
    [
      "sql",
      30
    ],
    [
      "nlp",
      29
    ],
    [
      "spark",
      29
    ]
  ]
}


In [42]:
# pip: install only what Colab does NOT already have. Check first:
import importlib.util
for pkg in ["numpy", "pandas", "sklearn", "torch", "rich"]:
    print(f"{pkg:<8}", "installed ✅" if importlib.util.find_spec(pkg) else "missing ❌  -> !pip install -q " + pkg)

numpy    installed ✅
pandas   installed ✅
sklearn  missing ❌  -> !pip install -q sklearn
torch    missing ❌  -> !pip install -q torch
rich     missing ❌  -> !pip install -q rich


## 7. 🎯 Try it yourself (in class, 5 minutes)

Write a function `skills_gap(my_skills, jobs, title)` that:
1. Takes a set of *your* skills, the list of job rows, and a job title (e.g. `"ML Engineer"`),
2. Counts how often each skill appears in postings **with that title** (a `Counter`),
3. Returns the top 5 skills that are demanded but **not in `my_skills`** — your personal study list.

Hints: filter with a comprehension, use `Counter`, subtract sets.

In [ ]:
def skills_gap(my_skills, jobs, title, top_n=5):
    # YOUR CODE HERE
    # 1. rows_for_title = [ ... ]
    # 2. demand = Counter( ... )
    # 3. return [(skill, n) for skill, n in demand.most_common() if ...][:top_n]
    pass

my_skills = {"python", "excel", "sql"}
print(skills_gap(my_skills, jobs, "ML Engineer"))

None


In [ ]:
def skills_gap(my_skills, jobs, title, top_n=5):
    # YOUR CODE HERE
    # 1. rows_for_title = [ ... ]
    # 2. demand = Counter( ... )
    # 3. return [(skill, n) for skill, n in demand.most_common() if ...][:top_n]
    pass

my_skills = {"python", "excel", "sql"}
print(skills_gap(my_skills, jobs, "ML Engineer"))

<details><summary>Solution</summary>

```python
def skills_gap(my_skills, jobs, title, top_n=5):
    rows_for_title = [r for r in jobs if r["title"] == title]
    demand = Counter(s for r in rows_for_title for s in r["skills"])
    return [(s, n) for s, n in demand.most_common() if s not in my_skills][:top_n]

print(skills_gap({"python", "excel", "sql"}, jobs, "ML Engineer"))
# e.g. [('pytorch', 61), ('llms', 58), ('docker', 55), ...]
```
</details>

## Homework starter 🏠

**Easy:** your skills dict → formatted table (one comprehension, one lambda, one f-string spec).
**Medium (job hunt):** save 3 real JDs as `jd1.txt`, `jd2.txt`, `jd3.txt` and count skill keywords.
**Stretch:** extend `jobs_tool.py` with `--city` filtering (done above — now add `--title` and write `report.json`).

In [1]:
# --- Easy: fill in your own numbers ---
my_confidence = {
    "python": 0.7,
    "flask": 0.7,
    "javascript": 0.6,
    "sql": 0.6,
    "mysql": 0.6,
    "react": 0.6,
    "rest_apis": 0.6,
    "git": 0.5,
    "oop": 0.5,
    "data_structures": 0.4,
    "algorithms": 0.4,
    "html_css": 0.6,
    "bootstrap": 0.5,
}
for skill, conf in sorted(my_confidence.items(), key=lambda kv: kv[1], reverse=True):
    print(f"{skill:<12}{conf:>5.0%}")

python        70%
flask         70%
javascript    60%
sql           60%
mysql         60%
react         60%
rest_apis     60%
html_css      60%
git           50%
oop           50%
bootstrap     50%
data_structures  40%
algorithms    40%


In [2]:
# --- Medium: JD keyword counter (job-hunt tool) ---
from collections import Counter

SKILLS = ["python", "sql", "pandas", "numpy", "pytorch", "tensorflow", "scikit-learn", "docker",
          "aws", "llm", "langchain", "fastapi", "git", "statistics", "nlp", "computer vision"]

def count_skills(path, skills=SKILLS):
    """Return Counter of how many times each skill string appears in the file (case-insensitive)."""
    try:
        with open(path, encoding="utf-8") as f:
            text = f.read().lower()
    except FileNotFoundError:
        print(f"(skipping {path}: not found — upload it first)")
        return Counter()
    return Counter({s: text.count(s) for s in skills if text.count(s) > 0})

# demo with a fake JD so the cell runs even before you upload real ones
with open("jd_demo.txt", "w") as f:
    f.write("We need strong Python and SQL. Experience with PyTorch or TensorFlow, Docker and AWS is a plus. Python, python, python!")

total = Counter()
for p in ["jd_demo.txt", "jd1.txt", "jd2.txt", "jd3.txt"]:
    c = count_skills(p)
    total += c
print("skills asked across all JDs:", total.most_common())
print("your study list:", sorted(set(total) - set(my_confidence)))

skills asked across all JDs: [('python', 11), ('sql', 5), ('aws', 4), ('pytorch', 3), ('docker', 3), ('pandas', 3), ('numpy', 2), ('nlp', 2), ('tensorflow', 1)]
your study list: ['aws', 'docker', 'nlp', 'numpy', 'pandas', 'pytorch', 'tensorflow']


## Key takeaways 🔑

- **Variables are stickers on objects.** `b = a` aliases; `a.copy()` copies. Mutable (list/dict/set/array) can change in place; immutable (int/str/tuple) cannot. Never use a mutable default argument.
- **Four containers:** list (grows), tuple (fixed record), dict (lookup by name), set (membership/unique).
- **Functions are objects** with keyword defaults — every library API is a function signature. Tomorrow we exploit "functions are objects" to build decorators.
- **Five idioms:** `f"{x:.4f}"`, `[expr for item in src if cond]`, `key=lambda`, `try/except SpecificError`, `with open(...)`.
- **Pipeline shape:** load → clean → aggregate → sort → report. Day 03 does this in pandas; today you built it by hand.
- **Modules:** any `.py` file; `if __name__ == "__main__":` lets a file be both script and library.

**Before Day 02:** push this notebook to your `ai-ml-45days` GitHub repo, and do the Medium homework — it becomes the seed of the resume↔JD matcher we build on Day 28.